# SV manuscript stats (Phase 1 / Phase 2)

Reads unified site tables + discovery TSVs from `sv_annotation` and renders:
1. Manuscript total-site comparison table
2. Ancestry-ordered Ebert-style cumulative discovery plots (frequency, region, CADD-SV)

Set the paths below to local downloads or GCS-fused paths.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Bootstrap scripts/ from $WORKSPACE_BUCKET/scripts/ when not on the VM.
for _d in (Path.cwd() / "scripts", Path.cwd().parent / "scripts"):
    if (_d / "terra_notebook.py").is_file():
        sys.path.insert(0, str(_d.resolve()))
        break
else:
    _bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
    if not _bucket:
        raise FileNotFoundError(
            "scripts/ not found locally and WORKSPACE_BUCKET is unset. "
            "Upload scripts/ to gs://WORKSPACE/scripts/."
        )
    _dest = (Path.cwd() / "scripts").resolve()
    _dest.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        ["gsutil", "-m", "rsync", "-r", f"{_bucket}/scripts/", str(_dest) + "/"]
    )
    sys.path.insert(0, str(_dest))

from terra_notebook import init_notebook

SCRIPTS = init_notebook(
    "workspace_paths.py",
    "merge_manuscript_counts.py",
    "plot_discovery.py",
)
from workspace_paths import sv_output_dir

import pandas as pd

OUT = sv_output_dir()

PHASE1_COUNTS = OUT / "aou_lr_phase1.manuscript_counts.tsv"
PHASE2_COUNTS = OUT / "aou_lr_phase2.manuscript_counts.tsv"
PHASE2_DISCOVERY = OUT / "aou_lr_phase2.discovery.tsv"
PHASE2_DISCOVERY_REGION = OUT / "aou_lr_phase2.discovery.region.tsv"
PHASE2_DISCOVERY_CADD = OUT / "aou_lr_phase2.discovery.cadd.tsv"
OUTDIR = OUT / "figures"
OUTDIR.mkdir(parents=True, exist_ok=True)

print("OUT:", OUT)
print("OUTDIR:", OUTDIR)


## Manuscript site-count table

In [ ]:
import subprocess

merged = OUTDIR.parent / "manuscript_sv_counts.merged.tsv"
if PHASE1_COUNTS.exists() and PHASE2_COUNTS.exists():
    subprocess.check_call([
        sys.executable,
        str(SCRIPTS / "merge_manuscript_counts.py"),
        "--phase1-tsv", str(PHASE1_COUNTS),
        "--phase2-tsv", str(PHASE2_COUNTS),
        "--out-tsv", str(merged),
    ])
    display(pd.read_csv(merged, sep="\t"))
elif PHASE2_COUNTS.exists():
    display(pd.read_csv(PHASE2_COUNTS, sep="\t"))
else:
    print("Count TSVs not found yet — point PHASE*_COUNTS at AnnotateSvCallset outputs.")

## Ebert-style discovery plots

Separate panels for Phase 1 and Phase 2 (not overlaid). Stratify by region and CADD-SV bins.

In [ ]:
def plot_if_exists(tsv: Path, png: Path, title: str):
    if not tsv.exists():
        print(f"Missing {tsv}")
        return
    subprocess.check_call([
        sys.executable,
        str(SCRIPTS / "plot_discovery.py"),
        "--discovery-tsv", str(tsv),
        "--out-png", str(png),
        "--title", title,
    ])
    print("Wrote", png)

plot_if_exists(PHASE2_DISCOVERY, OUTDIR / "phase2_discovery.png", "Phase 2 cumulative SV discovery")
plot_if_exists(PHASE2_DISCOVERY_REGION, OUTDIR / "phase2_discovery_region.png", "Phase 2 discovery by repeat context")
plot_if_exists(PHASE2_DISCOVERY_CADD, OUTDIR / "phase2_discovery_cadd.png", "Phase 2 discovery by CADD-SV bin")